# Araba Gövde Tipi Sınıflandırıcı - Colab Kurulum
**Sırayla çalıştırın. Her hücre tamamlanmadan bir sonrakine geçmeyin.**

In [ ]:
# HÜCRE 1: GPU kontrolü
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')

In [ ]:
# HÜCRE 2: Google Drive bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# HÜCRE 3: Proje dosyalarını çıkar (Drive'dan)
# ÖNCESİNDE: car_classifier_project.zip'i Google Drive'ın köküne yükleyin
import zipfile, os

zip_path = '/content/drive/MyDrive/car_classifier_project.zip'
if not os.path.exists(zip_path):
    print('HATA: Drive kökünde car_classifier_project.zip bulunamadı!')
    print('Lütfen önce Drive\'a yükleyin.')
else:
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print('Proje dosyaları çıkarıldı.')
    !ls /content/car_classifier_project/car_classifier/

In [ ]:
# HÜCRE 4: Dataset'i çıkar (Drive'dan)
# ÖNCESİNDE: dataset_6class.zip'i Google Drive'ın köküne yükleyin
import zipfile, os

dataset_zip = '/content/drive/MyDrive/dataset_6class.zip'
dest = '/content/car_classifier_project/car_classifier/'

if not os.path.exists(dataset_zip):
    print('HATA: Drive kökünde dataset_6class.zip bulunamadı!')
else:
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(dataset_zip, 'r') as z:
        z.extractall(dest)
    print('Dataset çıkarıldı.')
    for cls in sorted(os.listdir(dest + 'dataset')):
        n = len(os.listdir(os.path.join(dest + 'dataset', cls)))
        print(f'  {cls:<20}: {n}')

In [ ]:
# HÜCRE 5: Kaggle API kur
import os, json

# Kaggle kullanıcı adınızı ve API key'inizi buraya yazın:
KAGGLE_USERNAME = ""   # örn: "merva123"
KAGGLE_KEY      = ""   # token: KGAT_...

assert KAGGLE_USERNAME and KAGGLE_KEY, "USERNAME ve KEY boş bırakılamaz!"

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle API hazır.')

In [ ]:
# HÜCRE 6: Stanford Cars indir → STATION_WAGON topla
import shutil, os

dst_sw = '/content/car_classifier_project/car_classifier/dataset/STATION_WAGON'
os.makedirs(dst_sw, exist_ok=True)

# İndir
!kaggle datasets download -d jutrera/stanford-car-dataset-by-classes-folder -p /content/stanford_cars --unzip -q
print('Stanford Cars indirildi.')

# Train wagon
src_train = '/content/stanford_cars/car_data/car_data/train'
wagons = [c for c in os.listdir(src_train) if 'wagon' in c.lower()]
print('Wagon sınıfları:', wagons)
total = 0
for cls in wagons:
    for img in os.listdir(os.path.join(src_train, cls)):
        shutil.copy2(os.path.join(src_train, cls, img),
                     os.path.join(dst_sw, f'{cls}_{img}'))
        total += 1

# Test wagon
src_test = '/content/stanford_cars/car_data/car_data/test'
wagons_t = [c for c in os.listdir(src_test) if 'wagon' in c.lower()]
for cls in wagons_t:
    for img in os.listdir(os.path.join(src_test, cls)):
        shutil.copy2(os.path.join(src_test, cls, img),
                     os.path.join(dst_sw, f'test_{cls}_{img}'))
        total += 1

print(f'STATION_WAGON toplam: {len(os.listdir(dst_sw))} görüntü')

In [ ]:
# HÜCRE 7: MICRO topla (icrawler)
!pip install icrawler -q

from icrawler.builtin import BingImageCrawler
import os, shutil

dst_micro = '/content/car_classifier_project/car_classifier/dataset/MICRO'
os.makedirs(dst_micro, exist_ok=True)

SEARCHES = [
    ('Smart ForTwo city car side view', 80),
    ('Fiat 500 exterior car photo', 80),
    ('Kia Picanto exterior side view', 70),
    ('Toyota Aygo exterior side view', 70),
    ('Suzuki Alto micro car exterior', 70),
    ('Renault Twingo city car photo', 70),
    ('Volkswagen Up city car exterior', 70),
    ('Daihatsu Cuore small car', 60),
    ('Citroen C1 exterior side view', 70),
]

for i, (query, max_num) in enumerate(SEARCHES):
    tmpdir = f'/tmp/micro_tmp_{i}'
    os.makedirs(tmpdir, exist_ok=True)
    crawler = BingImageCrawler(storage={'root_dir': tmpdir})
    crawler.crawl(keyword=query, max_num=max_num,
                  filters={'type': 'photo', 'size': 'medium'})
    moved = 0
    for img in os.listdir(tmpdir):
        src = os.path.join(tmpdir, img)
        if os.path.isfile(src) and os.path.getsize(src) > 8000:
            shutil.move(src, os.path.join(dst_micro, f'micro_{i}_{img}'))
            moved += 1
    shutil.rmtree(tmpdir, ignore_errors=True)
    print(f"'{query}' → {moved} görüntü")

print(f'\nMICRO toplam: {len(os.listdir(dst_micro))} görüntü')

In [ ]:
# HÜCRE 8: Dataset özeti
import os
dataset_dir = '/content/car_classifier_project/car_classifier/dataset'
total = 0
print('Sınıf başına görüntü sayısı:')
for cls in sorted(os.listdir(dataset_dir)):
    n = len([f for f in os.listdir(os.path.join(dataset_dir, cls))
             if os.path.isfile(os.path.join(dataset_dir, cls, f))])
    total += n
    print(f'  {cls:<20}: {n}')
print(f'Toplam: {total} görüntü')
assert total > 15000, f'Eksik veri! Toplam: {total}'

In [ ]:
# HÜCRE 9: Eğitim başlat (T4 GPU, ~2-3 saat)
import os
os.chdir('/content/car_classifier_project/car_classifier')
!python -u train.py 2>&1 | tee /content/drive/MyDrive/train_log.txt

In [ ]:
# HÜCRE 10: Eğitim bitince modeli Drive'a kaydet
import shutil
shutil.copy(
    '/content/car_classifier_project/car_classifier/saved_model/car_body_classifier.pt',
    '/content/drive/MyDrive/car_body_classifier.pt'
)
shutil.copytree(
    '/content/car_classifier_project/car_classifier/plots',
    '/content/drive/MyDrive/plots',
    dirs_exist_ok=True
)
print('Model ve grafikler Drive\'a kaydedildi!')